# YOLOv8n — Damage Detection on Medicine Box

Trains **YOLOv8n** on your Roboflow dataset export and exports **ONNX** for on-device inference in the LabelCheck app.

**Before running:** `Runtime → Change runtime type → T4 GPU`.

Your dataset (v2): 183 train / 15 valid / 9 test, augmented in Roboflow (auto-orient, exposure, saturation, grayscale, brightness, rotation, h-flip).

In [ ]:
!nvidia-smi

## 1. Install

In [ ]:
%pip install -q ultralytics onnx onnxruntime
import ultralytics
ultralytics.checks()

## 2. Upload your dataset

Run the cell, then pick your Roboflow YOLOv8 export zip (e.g. `Damage detection on medicine box.yolov8.zip`).

For a large file you can instead mount Google Drive (`from google.colab import drive; drive.mount('/content/drive')`) and set `zip_name` to the file's path in Drive — skip the `files.upload()` line if you do.

In [ ]:
import os, zipfile
from google.colab import files

uploaded = files.upload()          # pick your .zip
zip_name = next(iter(uploaded))

extract_dir = '/content/dataset'
os.makedirs(extract_dir, exist_ok=True)
with zipfile.ZipFile(zip_name) as z:
    z.extractall(extract_dir)

print('Extracted:', os.listdir(extract_dir))

## 3. Locate and fix `data.yaml`

Rewrites the split paths to absolute so training finds the images regardless of working directory, and prints your class names.

In [ ]:
import glob, yaml

data_yaml = glob.glob('/content/dataset/**/data.yaml', recursive=True)[0]
root = os.path.dirname(data_yaml)

with open(data_yaml) as f:
    cfg = yaml.safe_load(f)

# Roboflow folder names: train / valid / test ; yaml keys: train / val / test
split_dirs = {'train': 'train', 'val': 'valid', 'test': 'test'}
for key, folder in split_dirs.items():
    p = os.path.join(root, folder, 'images')
    if os.path.isdir(p):
        cfg[key] = p
cfg['path'] = root

with open(data_yaml, 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print('data.yaml:', data_yaml)
print('classes  :', cfg.get('names'))

## 4. Train

YOLOv8n (nano) — small and fast, the right size to bundle into a phone app.

> Your Roboflow export already baked in augmentations, and Ultralytics adds mild augmentation of its own each epoch. That's fine for a small set. If it looks over-augmented, pass e.g. `degrees=0, fliplr=0, hsv_s=0` to tone it down.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')
model.train(
    data=data_yaml,
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,       # early-stop if val stops improving
    project='/content/runs',
    name='damage',
    seed=42,
)

## 5. Validate

In [ ]:
best_pt = '/content/runs/damage/weights/best.pt'
model = YOLO(best_pt)
metrics = model.val(data=data_yaml)
print('mAP50   :', round(float(metrics.box.map50), 4))
print('mAP50-95:', round(float(metrics.box.map), 4))

## 6. Export ONNX

`imgsz=640` must match the letterbox size the app preprocesses to. This exports raw detection output (no baked-in NMS) — the app does confidence filtering + NMS in Dart.

In [ ]:
onnx_path = model.export(format='onnx', imgsz=640, opset=12)
print('ONNX     :', onnx_path)
print('classes  :', model.names)   # index -> name; the app keys damage logic off these

## 7. Download

Save `best.onnx` (bundle into the app's `assets/`) and `best.pt` (keep as your source weights). **Note the class names printed above** — send them to me and I'll wire the on-device inference to match.

In [ ]:
from google.colab import files
files.download(onnx_path)
files.download(best_pt)